In [ ]:
!pip install -U pip
!pip install -U torch torchvision transformers datasets bitsandbytes accelerate peft
!sudo apt-get install -y cmake

In [ ]:
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [1]:
import torch
import transformers
import datasets
import bitsandbytes
import accelerate
import peft
import unsloth

torch.__version__, transformers.__version__, datasets.__version__, bitsandbytes.__version__, accelerate.__version__, peft.__version__

2025-02-22 19:49:32.516503: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-22 19:49:32.530672: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-02-22 19:49:32.548819: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-02-22 19:49:32.554414: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-22 19:49:32.567282: I tensorflow/core/platform/cpu_feature_guar

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


('2.6.0+cu124', '4.49.0', '3.3.2', '0.45.2', '1.4.0', '0.14.0')

# Get the model

In [2]:
max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # 4bit quantization to reduce memory usage

In [14]:
from unsloth import FastLanguageModel

model_names = [
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-3B-Instruct",
    "unsloth/Llama-3.2-1B-bnb-4bit",
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
]
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_names[0],
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.568 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [5]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0): LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), 

# Set LoRa adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # LoRa rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,  # Rank Stabilized LoRA
    loftq_config=None,
)

Unsloth 2025.2.15 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


In [7]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear4b

# Get the dataset

In [8]:
from datasets import load_dataset

dataset = load_dataset("Tom158/Nutritional-LLama", split="train")

In [9]:
dataset.column_names

['System', 'User', 'Nutritionist', 'text']

In [10]:
dataset[0]["System"]

'You serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.'

In [11]:
dataset[0]["User"]

"I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?"

In [12]:
dataset[0]["Nutritionist"]

"Ground lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime."

In [13]:
dataset[0]["text"]

"<s>[INST] <<SYS>>\nYou serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.\n<</SYS>>\n\nI've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight? [/INST] Ground lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime. </s>"

In [14]:
dataset = dataset.rename_column("text", "text_llama_2")

In [15]:
dataset.column_names

['System', 'User', 'Nutritionist', 'text_llama_2']

In [16]:
tokenizer.eos_token

'<|eot_id|>'

In [17]:
def format_prompt(example):
    system = example["System"]
    user = example["User"]
    assistant = example["Nutritionist"]

    llama3_1_prompt = f"""\
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
{system}<|eot_id|>

<|start_header_id|>user<|end_header_id|>
{user}<|eot_id|>

<|start_header_id|>assistant<|end_header_id|>
{assistant}<|eot_id|>\
"""
    return {"text": llama3_1_prompt}
    

dataset = dataset.map(format_prompt)

In [18]:
dataset.column_names

['System', 'User', 'Nutritionist', 'text_llama_2', 'text']

In [19]:
dataset[0]["text"]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.<|eot_id|>\n\n<|start_header_id|>user<|end_header_id|>\nI've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|eot_id|>\n\n<|start_header_id|>assistant<|end_header_id|>\nGround lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate t

# Train the model

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        # num_train_epochs=1, # Set this for 1 full training run.
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="steps",
        save_steps=50,
        report_to="none",
    ),
)

In [ ]:
# Mask train on the assistant outputs and ignore the loss on the user's inputs
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n",
)

In [22]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

"<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\nYou serve as a professional nutrition advisor and a friendly assistant which helps the user to improvise the personal health. Ensure that interaction is natural, friendly and human-like.<|eot_id|>\n\n<|start_header_id|>user<|end_header_id|>\nI've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|eot_id|>\n\n<|start_header_id|>assistant<|end_header_id|>\nGround lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your 

In [23]:
space = tokenizer(" ", add_special_tokens=False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[0]["labels"]])

"                                                                                                \nGround lean can be a great protein source for you, but considering your current health situation, it's essential to balance it out with other nutrient-dense foods. Since ground lean is high in fat, especially saturated fat, try pairing it with more fiber-rich options like whole grain toast or veggies to help offset the calorie intake. Also, consider reducing the overall portion size and adding some complex carbs to your breakfast plate to keep you full till lunchtime.<|eot_id|>"

In [24]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.568 GB.
1.086 GB of memory reserved.


In [25]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 1,488 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 5,636,096


Step,Training Loss
1,1.743100
2,1.686500
3,1.554100
4,1.383700
5,1.435800
6,1.313100
7,1.195100
8,1.240800
9,1.237600
10,1.215200


In [26]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

63.1786 seconds used for training.
Peak reserved memory = 3.689 GB.
Peak reserved memory for training = 2.603 GB.
Peak reserved memory % of max memory = 25.323 %.
Peak reserved memory for training % of max memory = 17.868 %.


# Save the fine-tuned model

In [ ]:
model.save_pretrained("nutritionalist")
tokenizer.save_pretrained("nutritionalist")

In [22]:
model.push_to_hub("noroozi/nutritionalist-llama-3.2", token=HF_TOKEN)
tokenizer.push_to_hub("noroozi/nutritionalist-llama-3.2", token=HF_TOKEN)

README.md:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Saved model to https://huggingface.co/noroozi/nutritionalist-llama-3.2


tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

### gguf

In [20]:
model.save_pretrained_gguf("nutritionalist-gguf", tokenizer, quantization_method=["q4_k_m", "q8_0", "q5_k_m",])

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 17.75 out of 30.89 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 16/16 [00:00<00:00, 387.38it/s]

Unsloth: Saving tokenizer...

 Done.
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m', 'q8_0', 'q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at gguf into f16 GGUF format.
The output location will be /home/sagemaker-user/gguf/unsloth.F16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: gguf
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:gguf: loading model part 'model.safetensors'
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> F16, shape = {2048, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    

In [24]:
model.push_to_hub_gguf(
        "noroozi/nutritionalist-llama-3.2",
        tokenizer,
        quantization_method=["q4_k_m", "q8_0", "q5_k_m",],
        token=HF_TOKEN,
    )

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 17.56 out of 30.89 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 16/16 [00:00<00:00, 390.77it/s]

Unsloth: Saving tokenizer...

 Done.
Done.
==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m', 'q8_0', 'q5_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at noroozi/nutritionalist-llama-3.2 into f16 GGUF format.
The output location will be /home/sagemaker-user/noroozi/nutritionalist-llama-3.2/unsloth.F16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: nutritionalist-llama-3.2
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:gguf: loading model part 'model.safetensors'
INFO:hf-to-gguf:token_embd.weight,           

unsloth.Q4_K_M.gguf:   0%|          | 0.00/808M [00:00<?, ?B/s]

...

No files have been modified since last commit. Skipping to prevent empty commit.


Saved GGUF to https://huggingface.co/noroozi/nutritionalist-llama-3.2
Unsloth: Uploading GGUF to Huggingface Hub...


unsloth.Q5_K_M.gguf:   0%|          | 0.00/912M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Saved GGUF to https://huggingface.co/noroozi/nutritionalist-llama-3.2


# Load the model

In [27]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="safetensors",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.568 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


# Inference

In [29]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)
# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

user_query = "I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?"
messages = [
    {"role": "user", "content": user_query},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True,
                         temperature=1.5, min_p=0.1)
tokenizer.batch_decode(outputs)

["<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nI've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nConsidering your weight and age, it's great that you're focusing on a balanced diet! Ground lean can be a tasty addition to your breakfast, providing you with a good amount of protein and fiber. As someone who weighs over 105 kg (including the current weight), it's essential to prioritize healthy habits. To support"]

In [ ]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=128,
                   use_cache=True, temperature=1.5, min_p=0.1)

# Ollama Inference

In [31]:
print(tokenizer._ollama_modelfile)


FROM {__FILE_LOCATION__}
TEMPLATE """{{ if .Messages }}
{{- if or .System .Tools }}<|start_header_id|>system<|end_header_id|>
{{- if .System }}

{{ .System }}
{{- end }}
{{- if .Tools }}

You are a helpful assistant with tool calling capabilities. When you receive a tool call response, use the output to format an answer to the original use question.
{{- end }}
{{- end }}<|eot_id|>
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if eq .Role "user" }}<|start_header_id|>user<|end_header_id|>
{{- if and $.Tools $last }}

Given the following functions, please respond with a JSON for a function call with its proper arguments that best answers the given prompt.

Respond in the format {"name": function name, "parameters": dictionary of argument name and its value}. Do not use variables.

{{ $.Tools }}
{{- end }}

{{ .Content }}<|eot_id|>{{ if $last }}<|start_header_id|>assistant<|end_header_id|>

{{ end }}
{{- else if eq .Role "assistant" }}<|start_header

In [ ]:
!ollama create nutritionalist-llama3.2 -f ./gguf-f16/Modelfile

In [ ]:
!curl http://localhost:11434/api/chat -d '{ \
    "model": "ollama_model", \
    "messages": [ \
        { "role": "user", "content": "I've been trying to lose some weight and get in shape, but I love ground lean as part of my breakfast routine. Is it okay to include this in my diet considering I'm 43 and 109kg with overweight?" } \
    ] \
    }'